# Step 10b-X: sinh query mở rộng cho 4 bộ mới, có lưu logprobs (GPU)

Cùng thiết lập với Step 1C: LLaMA 3.1-8B-Instruct, vLLM fp16, TP=2, greedy, 256 token, prompt Phụ lục G. Điểm mới duy nhất là `--logprobs 5`, dùng sau này cho phân tích độ tự tin; văn bản sinh ra không đổi.

- **Accelerator:** GPU T4 ×2. **Internet:** On. **Secret:** `HF_TOKEN` (LLaMA là mô hình gated, như Step 1C).
- **Input:** `qudar-beir-data-2`, `qudar-patches` (bản mới).
- **Thời gian ước tính:** 21.088 query ở khoảng 5–6 query/giây → khoảng 65–75 phút, cộng 3 lần nạp mô hình × 3–4 phút. Tổng khoảng 1,5 giờ.
- **Sau khi chạy:** tạo dataset `qudar-expansions-2` từ thư mục `expansions/` (gồm file JSON và thư mục `logprobs/`).
- Bị ngắt giữa chừng: chạy lại các ô, phần đã sinh sẽ được bỏ qua.

In [ ]:
import glob, os
DATA2 = os.path.dirname(os.path.dirname(glob.glob("/kaggle/input/**/qudar-beir-data-2/**/dbpedia-entity/corpus.jsonl", recursive=True)[0]))
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
PATCH = os.path.dirname(glob.glob("/kaggle/input/**/expand_queries.py", recursive=True)[0])
assert "--logprobs" in open(f"{PATCH}/expand_queries.py").read(), "qudar-patches chưa phải bản mới"
!pip install -q vllm
!pip install -q --no-deps beir
E = f"python {PATCH}/expand_queries.py --data-root {DATA2} --out-dir /kaggle/working/expansions --logprobs 5"

In [ ]:
# Test của 4 bộ: 300 + 323 + 6666 + 400 = 7689 query
!{E} --datasets scifact nfcorpus fever dbpedia-entity --split test

In [ ]:
# Train: SciFact 809 + NFCorpus 2590
!{E} --datasets scifact nfcorpus --split train

In [ ]:
# Train FEVER: 10.000 query đầu (như HotpotQA)
!{E} --datasets fever --split train --max-queries 10000

In [ ]:
# Tóm tắt để gửi lại
import json, numpy as np
from pathlib import Path
base = Path("/kaggle/working/expansions")
expect = {"scifact_test": 300, "scifact_train": 809, "nfcorpus_test": 323, "nfcorpus_train": 2590,
          "fever_test": 6666, "fever_train": 10000, "dbpedia-entity_test": 400}
for key, n in expect.items():
    ds, split = key.rsplit("_", 1)
    exp = json.loads((base / f"expanded_queries_{ds}_{split}_expansion.json").read_text())
    words = [len(v.split()) for v in exp.values()]
    parts = sorted((base / "logprobs" / key).glob("part_*.npz"))
    lp_q = set(); toks = 0; mean_lp = []
    for p in parts:
        z = np.load(p)
        lp_q.update(z["qids"].tolist()); toks += len(z["token_ids"]); mean_lp.append(float(np.nanmean(z["chosen_lp"])))
    print(f"{key}: {len(exp)}/{n} expansion, {sum(w == 0 for w in words)} rỗng, {np.mean(words):.0f} từ TB | "
          f"logprobs: {len(lp_q)} query, {toks} token, log-prob TB {np.mean(mean_lp):.3f}")
!du -sh /kaggle/working/expansions